# 03 - Train K-Nearest Neighbors (KNN)

Train KNN classifier for molecular toxicity prediction using ECFP4 fingerprints.

**Key Features:**
- Uses ECFP4 (2048-bit) fingerprints for molecular encoding
- Grid search for optimal k value
- Cross-validation for robust evaluation

In [1]:
# ============================================================================
# IMPORTS
# ============================================================================
import os
import sys
import numpy as np
import pandas as pd
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('../src'))
print("✓ Imports complete")

✓ Imports complete


In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================
# KNN hyperparameter search space
PARAM_GRID = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

TOXICITY_ENDPOINTS = [
    'NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

MODELS_DIR = '../models/baseline_models'
RESULTS_DIR = '../results/baseline_models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"KNN Parameters to search: {sum(len(v) for v in PARAM_GRID.values())} options")

KNN Parameters to search: 12 options


In [3]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================
def train_knn(toxicity_name):
    """Train KNN for a single toxicity endpoint."""
    print(f"\n{'='*60}")
    print(f"Training KNN for {toxicity_name}")
    print(f"{'='*60}")
    
    # Load preprocessed data
    cache_path = f'../Data/cache/{toxicity_name}/splits.pkl'
    if not os.path.exists(cache_path):
        print(f"⚠️ Data not found. Run preprocessing first.")
        return None
    
    with open(cache_path, 'rb') as f:
        data = pickle.load(f)
    
    X_train, y_train = data['train']['X'], data['train']['y']
    X_val, y_val = data['val']['X'], data['val']['y']
    X_test, y_test = data['test']['X'], data['test']['y']
    
    # Combine train and val for grid search
    X_train_val = np.vstack([X_train, X_val])
    y_train_val = np.concatenate([y_train, y_val])
    
    print(f"Train+Val: {len(X_train_val)}, Test: {len(X_test)}")
    
    # Grid search
    knn = KNeighborsClassifier()
    grid_search = GridSearchCV(knn, PARAM_GRID, cv=5, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train_val, y_train_val)
    
    print(f"Best params: {grid_search.best_params_}")
    print(f"CV ROC-AUC: {grid_search.best_score_:.4f}")
    
    # Evaluate on test set
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]
    
    test_auc = roc_auc_score(y_test, y_proba)
    test_acc = accuracy_score(y_test, y_pred)
    
    print(f"Test ROC-AUC: {test_auc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    
    # Save model
    model_dir = os.path.join(MODELS_DIR, toxicity_name)
    os.makedirs(model_dir, exist_ok=True)
    with open(os.path.join(model_dir, 'KNN_model.pkl'), 'wb') as f:
        pickle.dump({'model': best_model, 'params': grid_search.best_params_}, f)
    
    return {'model': best_model, 'test_auc': test_auc, 'test_acc': test_acc}

# Train for one endpoint
result = train_knn('NR-AhR')


Training KNN for NR-AhR
Train+Val: 5895, Test: 654


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# TRAIN ALL ENDPOINTS (uncomment to run)
# ============================================================================
# all_results = {}
# for endpoint in TOXICITY_ENDPOINTS:
#     all_results[endpoint] = train_knn(endpoint)
# 
# # Summary
# summary_df = pd.DataFrame([
#     {'Endpoint': k, 'Test_AUC': v['test_auc'], 'Test_Acc': v['test_acc']}
#     for k, v in all_results.items() if v
# ])
# print(summary_df)